# Why LangGraph Exists [Step 1 - LCEL Limitations and Graph Fundamentals]

> **MLCourse - Agentic AI - LangGraph**

This notebook motivates why LangGraph was created by showing where
traditional LCEL chains break down and how a directed graph model
solves those problems. We build a simple chain, identify its limits,
and then introduce the core StateGraph concept.

### Import core libraries needed throughout the notebook.


In [ ]:
import os                          # Environment variable access
import operator                    # Used for operator.add in reducers later
from typing import TypedDict, Annotated, Sequence  # Type hints for state
from dotenv import load_dotenv     # Load API keys from .env file

load_dotenv()                      # Load .env into environment


### Show that API keys are available (green) or warn (red).


In [ ]:
# ChatOllama does not need a key, so this notebook always runs green.
api_key = os.environ.get("OPENAI_API_KEY", "")
if api_key:
    print("[GREEN] API key found -- optional LLM calls will work")
else:
    print("[GREEN] No API key needed -- using local ChatOllama")


### Part 1: A Simple LCEL Chain


In [ ]:
# LCEL (LangChain Expression Language) pipes data through a sequence
# of steps. This works well for linear, acyclic workflows.

from langchain_core.prompts import ChatPromptTemplate          # Prompt templating
from langchain_ollama import ChatOllama                        # Local LLM

llm = ChatOllama(model="llama3.1:8b", temperature=0)           # Local model

# Build a simple prompt chain: template -> llm
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    ("user", "{input}")
])

# LCEL pipe: prompt | llm
chain = prompt | llm                                            # Linear pipeline

# Invoke the chain with a simple question
result = chain.invoke({"input": "What is LangGraph in one sentence?"})
print("Chain result:")
print(result.content)


### Part 2: Where Chains Break -- Cyclic Workflows


In [ ]:
# A simple chain is a straight line: input -> process -> output.
# But many real agent workflows need loops:
#   - A node checks a condition and decides whether to retry
#   - A tool-calling agent loops until the LLM stops calling tools
#   - A human-in-the-loop workflow pauses, resumes, and branches
#
# LCEL pipes are DAGs (directed acyclic graphs). They cannot represent
# cycles. Let us demonstrate the limitation conceptually.

# This would be the desired workflow for an agent with tool use:
#   [Classify] -> [Call Tool] -> [Check Result] --retry--> [Call Tool]
#                                        |
#                                      [Done]
#
# A linear chain cannot express the "retry" arrow because it would
# create a cycle. LCEL pipes are strictly forward-only.

print("LCEL chain type:", type(chain))
print("A chain is a RunnableSequence -- a linear pipeline with no loops.")
print()
print("Problems that LCEL cannot solve:")
print("  1. Cyclic workflows (agent loops until done)")
print("  2. Conditional branching with merge points")
print("  3. Human-in-the-loop pause/resume")
print("  4. Stateful multi-step reasoning with backtracking")


### Part 3: The Graph Solution


In [ ]:
# LangGraph models workflows as directed graphs with:
#   - **State**: a shared data structure (TypedDict) that flows through nodes
#   - **Nodes**: Python functions that read state and return partial updates
#   - **Edges**: connections between nodes, including conditional branches
#
# This graph model naturally supports cycles, branches, and state.

from langgraph.graph import StateGraph, START, END              # Core graph classes

# Define a simple state schema using TypedDict.
# TypedDict gives us type hints and a clear contract for what
# data flows through the graph.
class GraphState(TypedDict):
    input: str                  # The user's question
    output: str                 # The LLM's response

# Define a node: a function that takes state and returns a dict update.
# The returned dict is merged into the state (partial update pattern).
def generate(state: GraphState) -> dict:
    """Call the LLM to produce an answer."""
    response = chain.invoke({"input": state["input"]})  # Use the chain
    return {"output": response.content}                   # Partial state update

# Build a minimal graph with one node.
graph = StateGraph(GraphState)       # Create graph with our state schema
graph.add_node("generate", generate) # Register the node
graph.add_edge(START, "generate")    # Start -> generate
graph.add_edge("generate", END)      # generate -> End

# Compile the graph into a runnable object.
app = graph.compile()                # Freezes the graph for execution


### Part 4: Visualize the Graph


In [ ]:
# LangGraph can render the graph as a Mermaid diagram.

from IPython.display import Image, display                 # Notebook display

try:
    # draw_mermaid_png returns PNG bytes of the graph structure
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    # Fallback if Mermaid rendering is unavailable (no internet, etc.)
    print("Graph visualization unavailable in this environment.")
    print("Graph structure: START -> generate -> END")


### Part 5: Invoke the Graph


In [ ]:
# The compiled graph is callable like a chain, but with graph semantics.

final_state = app.invoke({"input": "What is LangGraph?", "output": ""})
print("Input:", final_state["input"])
print("Output:", final_state["output"])


### Part 6: Why Graphs Win


In [ ]:
# Even this trivial example hints at the power:
#
# 1. **Explicit control flow** -- edges are named, inspectable, debuggable
# 2. **State is first-class** -- every node sees and modifies shared state
# 3. **Cycles are natural** -- add an edge from any node back to any node
# 4. **Conditional routing** -- edges can depend on state values
# 5. **Visualization** -- the graph structure is renderable and documentable
#
# In the next notebooks we will explore state, nodes, edges, and
# conditional routing in detail.

print("Summary:")
print("  LCEL chains = linear pipelines (fast, simple)")
print("  LangGraph   = directed state graphs (cyclic, branching, stateful)")
print("  Use graphs when your workflow has loops, branches, or shared state.")
